<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/04_agentic_ai.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Async Python for AI Agents</h2>
            <h4 style="color:#00bf00;">Copyright 2026 Dr Elena Kramer, Braude College, Israel.</h4>
            <span style="color:#00bfff;">The OpenAI Agents SDK is asynchronous: every <code>Runner.run</code> must be awaited. In this lab, we learn what <code>async</code> and <code>await</code> really mean, first with simple examples without an LLM, and then with agents running on GPT-OSS:120B through Ollama.</span>
        </td>
    </tr>
</table>

Set `OLLAMA_BASE_URL` in the project-root `.env` file, and run the notebook from top to bottom.

## What you will learn

- What a **coroutine** is, and why calling an `async def` function does not run it.
- How `await` and `asyncio.gather` let us wait for several things **at the same time**.
- A common trap: blocking code inside `async def`.
- Why we write `await` directly in Jupyter, but `asyncio.run` in a Python script.
- How to run several **agents** in parallel, and how to set a timeout.

In [ ]:
import os
import time
import asyncio

from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, set_default_openai_client, set_default_openai_api, set_tracing_disabled

## Setup - the model

The same connection to Ollama as in the previous labs, with one change: the Agents SDK is asynchronous, so we use `AsyncOpenAI` instead of `OpenAI`. Then we tell the SDK to use this client for all agents, with the Chat Completions API that Ollama supports.

We don't need traces in this lab, so we turn tracing off.

In [ ]:
load_dotenv(override=True)
MODEL = "gpt-oss:120b"
# Always remember to do this!
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not OLLAMA_BASE_URL:
    raise ValueError("Set OLLAMA_BASE_URL in the project-root .env file")

openai = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)
set_default_openai_client(openai, use_for_tracing=False)
set_default_openai_api("chat_completions")

set_tracing_disabled(True)

# Part 1: async without an LLM

An agent spends most of its time **waiting** - for the model, for a web service, for a file.  
To keep things fast and predictable, we first simulate waiting with `asyncio.sleep`.

Our example: making breakfast. Coffee takes 3 seconds, toast 2 seconds, eggs 1 second.

## Step 1 - a coroutine

A function defined with `async def` is a **coroutine function**. Calling it does **not** run it - it only creates a coroutine object.  
`await` is what actually runs it.

In [ ]:
async def make(food, seconds):
    print(f"Start {food}")
    await asyncio.sleep(seconds)      # waiting, like waiting for the LLM
    print(f"{food} is ready")
    return food

In [ ]:
coffee = make("coffee", 3)
print(coffee)          # nothing happened yet - just a coroutine object

In [ ]:
result = await coffee  # now it runs
print(result)

## Step 2 - one after the other vs. at the same time

With `await` one by one, we wait for each dish to finish before starting the next: 3 + 2 + 1 = 6 seconds.

`asyncio.gather` starts all of them together and waits for all of them: about 3 seconds - the longest one.  
It returns the results **in the order we passed them**.

In [ ]:
start = time.time()
await make("coffee", 3)
await make("toast", 2)
await make("eggs", 1)
print(f"One after the other: {time.time() - start:.1f} seconds")

In [ ]:
start = time.time()
breakfast = await asyncio.gather(make("coffee", 3), make("toast", 2), make("eggs", 1))
print(breakfast)
print(f"With gather: {time.time() - start:.1f} seconds")

## Step 3 - a common trap: blocking code

`time.sleep` (and also `requests.get`, or reading a big file) **blocks** Python completely. While it blocks, nothing else can run - so `gather` doesn't help.

So inside `async def` we always wait with `await asyncio.sleep`, never with `time.sleep`.

In [ ]:
async def make_blocking(food, seconds):
    time.sleep(seconds)               # blocks everything!
    return food

start = time.time()
await asyncio.gather(make_blocking("coffee", 3), make_blocking("toast", 2), make_blocking("eggs", 1))
print(f"Blocking: {time.time() - start:.1f} seconds")

## Step 4 - Jupyter vs. a Python script

Jupyter already runs an **event loop** (the engine that runs coroutines), so we can write `await` directly in a cell.

In a regular `.py` file there is no running loop. We put our code in `async def main()` and start the loop with `asyncio.run`:

```python
import asyncio

async def main():
    breakfast = await asyncio.gather(make("coffee", 3), make("toast", 2))
    print(breakfast)

asyncio.run(main())
```

Don't call `asyncio.run` inside Jupyter - you will get an error, because a loop is already running.

# Part 2: async with agents

`Runner.run` is a coroutine, just like `make`. Everything from Part 1 works with agents too.

We use a short-answer agent, so the examples run quickly.

In [ ]:
agent = Agent(
    name="Explainer",
    instructions="You explain AI concepts in one short sentence.",
    model=MODEL,
)

## Step 1 - forgetting `await`

The most common mistake. Without `await` we get a coroutine object, not an answer.

In [ ]:
result = Runner.run(agent, "What is an AI agent?")
print(result)            # a coroutine - the agent did not run!

In [ ]:
result = await result    # now it runs
print(result.final_output)

## Step 2 - several agents in parallel with `gather`

Three questions, one after the other and then at the same time.

In [ ]:
start = time.time()
r1 = await Runner.run(agent, "What is a tool?")
r2 = await Runner.run(agent, "What is an agent loop?")
r3 = await Runner.run(agent, "What is a trace?")
print(f"One after the other: {time.time() - start:.1f} seconds")

In [ ]:
start = time.time()
r1, r2, r3 = await asyncio.gather(
    Runner.run(agent, "What is a tool?"),
    Runner.run(agent, "What is an agent loop?"),
    Runner.run(agent, "What is a trace?"),
)
print(f"In parallel: {time.time() - start:.1f} seconds")

print(r1.final_output)
print(r2.final_output)
print(r3.final_output)

## Step 3 - a timeout with `wait_for`

A slow answer should not freeze our application. `asyncio.wait_for` stops the run if it takes too long.  
Try changing the timeout.

In [ ]:
try:
    result = await asyncio.wait_for(Runner.run(agent, "What is RAG?"), timeout=2)
    print(result.final_output)
except asyncio.TimeoutError:
    print("Too slow - timed out")

# Congratulations!

You now know what `async` and `await` really do: coroutines, `gather`, the blocking trap, and how to run agents in parallel and with a timeout.

From now on, every `await Runner.run(...)` should make sense.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Create two agents with different personalities (for example a serious lecturer and a stand-up comedian). Ask both the same question at the same time with <code>asyncio.gather</code> and print both answers. Measure the time and compare it with asking them one after the other.</span>
        </td>
    </tr>
</table>